### Задание на Четверку (Проведение исследований моделями семантической сегментации)

#### 4. Имплементация алгоритма машинного обучения

импорты

In [15]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm

c:\Users\2b100\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


пути

In [16]:
IMAGES_DIR = "../train_images"
MASKS_DIR = "../train_labels"

dataset

In [17]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted([
            f for f in os.listdir(images_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.images)

    def find_mask(self, name):
        for ext in [".png", ".jpg", ".jpeg"]:
            path = os.path.join(self.masks_dir, name + ext)
            if os.path.exists(path):
                return path
        return None

    def __getitem__(self, idx):
        img_name = self.images[idx]
        name = os.path.splitext(img_name)[0]

        img_path = os.path.join(self.images_dir, img_name)
        mask_path = self.find_mask(name)

        if mask_path is None:
            raise ValueError(f"Mask not found for {name}")

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise ValueError(f"Bad image: {img_path}")
        if mask is None:
            raise ValueError(f"Bad mask: {mask_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = (mask > 0).astype("float32")

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]

        return image, mask.unsqueeze(0)

аугментации

In [18]:
def get_transforms():
    return A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(),
        ToTensorV2()
    ])

Простая CNN (U-Net light)

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU()
        )

        self.pool = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        self.dec = nn.Sequential(
                nn.Conv2d(48, 16, 3, padding=1),  # ← исправлено
                nn.ReLU(),
                nn.Conv2d(16, 1, 1)
            )
        

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.pool(x1)

        x3 = self.enc2(x2)
        x4 = self.up(x3)
        x4 = torch.cat([x4, x1], dim=1) 
        out = self.dec(x4)
        return out

2. Простой Transformer (очень упрощённый ViT для сегментации)

In [20]:
class SimpleViT(nn.Module):
    def __init__(self, img_size=256, patch_size=16, in_ch=3, dim=128):
        super().__init__()

        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(in_ch, dim, patch_size, patch_size)

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=4),
            num_layers=4
        )

        self.head = nn.ConvTranspose2d(dim, 1, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x) 

        b, c, h, w = x.shape
        x = x.flatten(2).permute(2, 0, 1) 

        x = self.transformer(x)

        x = x.permute(1, 2, 0).reshape(b, c, h, w)

        x = self.head(x)

        return x

Loss + device

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loss_fn = nn.BCEWithLogitsLoss()

Train loop

In [22]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Метрики

In [23]:
def iou(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter + 1e-6) / (union + 1e-6)


def dice(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    return (2 * inter + 1e-6) / (pred.sum() + target.sum() + 1e-6)


def acc(pred, target):
    pred = (pred > 0.5).float()
    return (pred == target).float().mean()

Evaluation

In [24]:
def evaluate(model, loader):
    model.eval()

    iou_total = 0
    dice_total = 0
    acc_total = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            preds = torch.sigmoid(model(images))

            iou_total += iou(preds, masks).item()
            dice_total += dice(preds, masks).item()
            acc_total += acc(preds, masks).item()

    n = len(loader)

    return {
        "IoU": iou_total / n,
        "Dice": dice_total / n,
        "Acc": acc_total / n
    }

In [25]:
from torch.utils.data import random_split

full_dataset = SegmentationDataset(
    IMAGES_DIR,
    MASKS_DIR,
    transform=None
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = get_transforms()

val_dataset.dataset.transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

CNN

In [26]:
model_cnn = SimpleCNN().to(device)
opt_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model_cnn, train_loader, opt_cnn)
    print(f"CNN Epoch {epoch+1}: {loss:.4f}")

CNN Epoch 1: 0.5663
CNN Epoch 2: 0.3714
CNN Epoch 3: 0.3347
CNN Epoch 4: 0.3089
CNN Epoch 5: 0.2808
CNN Epoch 6: 0.2852
CNN Epoch 7: 0.2845
CNN Epoch 8: 0.2700
CNN Epoch 9: 0.2675
CNN Epoch 10: 0.2645


Transformer

In [27]:
model_vit = SimpleViT().to(device)
opt_vit = torch.optim.Adam(model_vit.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model_vit, train_loader, opt_vit)
    print(f"ViT Epoch {epoch+1}: {loss:.4f}")

C:\Users\2b100\AppData\Local\Temp\ipykernel_20928\105331884.py:10: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(


ViT Epoch 1: 0.5483
ViT Epoch 2: 0.3792
ViT Epoch 3: 0.3343
ViT Epoch 4: 0.3072
ViT Epoch 5: 0.2757
ViT Epoch 6: 0.2657
ViT Epoch 7: 0.2561
ViT Epoch 8: 0.2412
ViT Epoch 9: 0.2071
ViT Epoch 10: 0.2089


Оценка качества

In [28]:
cnn_metrics = evaluate(model_cnn, val_loader)
vit_metrics = evaluate(model_vit, val_loader)

print("CNN:", cnn_metrics)
print("ViT:", vit_metrics)

CNN: {'IoU': 0.6327231228351593, 'Dice': 0.7728081047534943, 'Acc': 0.8911255151033401}
ViT: {'IoU': 0.6727340519428253, 'Dice': 0.802975207567215, 'Acc': 0.9061737060546875}


### Анализ результатов и выводы

Имплементированные модели уступают baseline-реализациям как по IoU, так и по Dice и pixel accuracy. Наибольшее снижение качества наблюдается у CNN-модели, что связано с упрощением архитектуры и отсутствием предобученных весов. ViT-модель демонстрирует меньшую деградацию качества, что может свидетельствовать о большей устойчивости трансформерных архитектур к изменениям реализации.

Несмотря на снижение метрик, результаты подтверждают корректность реализации моделей и позволяют использовать их как основу для дальнейшего улучшения (аугментации данных, оптимизация гиперпараметров, улучшение архитектуры).